# Transformace příznaků
Úloha zaměřená na implementaci metod PCA a LDA pro transformaci příznaků. 

Výchozí motivace
- Cílem je provést dekorelaci příznaků a vybrat pouze ty nejvýznamnější

Chceme
- Snížit zátěž (výpočetní, časovou) vlastního klasifikačního procesu.
- Zvýšit úspěšnost klasifikace

## Data

In [1]:
import numpy as np
import usu

npzfile = np.load('data/data_12.npz')
npzfile.files


['testData', 'testRef', 'trainData', 'trainRef']

In [2]:
testData = npzfile['testData']
testRef = npzfile['testRef']

trainData = npzfile['trainData']
trainRef = npzfile['trainRef']

trainData.shape,trainRef.shape, testData.shape, testRef.shape

((1050, 5), (1050, 1), (850, 5), (850, 1))

### Výpočet úspěšnosti
$$ accuracy = \frac{\text{počet správně klasifikovaných objektů}}{\text{počet všech klasifikovaných  objektů}} $$

In [3]:
def accuracy(testRef, predRef):
    """
    vraci uspesnost v procentech
    """
    return (predRef==testRef).mean() * 100


### Vzdálenostní funkce
V dané uloze implemetujeme jenom euklidovskou vzdálenost

#### Euklidovská vzdálenost (L2)
$$ d(x,z) = \sqrt{\sum_{i=0}^{Dim}{(x_i - z_i)^2}} $$

Při implementaci jde vynechat operaci druhé odmocniny, protože druhá odmocnina je monotónní rostoucí funkce. To znamená, že se mění jenom absolutní hodnoty vzdálenosti, ale pořadí se zachovává:
$$ d(x,z) = \sum_{i=0}^{Dim}{(x_i - z_i)^2} $$

In [4]:
def euclidian_distance (testItem, trainData):
    """
    vypocet vzdalenosti jendoho testovaciho vzorku ke vsem trenovacim datum
    """
    distances = np.sum((trainData - testItem) ** 2, axis=1)
    
    return distances

euclidian_distance(testData[0],trainData)

array([38.54881076, 54.30823974, 56.90800423, ..., 39.14515371,
       20.25885791, 17.23315459])

In [14]:
def getPrediction(trainData, trainRef, testData):
    
    # print(trainData.shape, trainRef.shape, testData.shape)
        
    #pomoci funkce euclidian_distance
    
    nTestElements = np.size(testData,0)
    nClasses = np.size(np.unique(trainRef),0) 
    predRef = np.zeros([nTestElements,1])
    
    #pomoci etalonu
    
    etalons = np.zeros([nClasses, trainData.shape[1]])
    for i in range(nClasses):
        etalons[i] = np.mean(trainData[trainRef[:,0]==i], axis=0)
    
    for i in range(nTestElements):
        distances = euclidian_distance(testData[i], etalons)
        predRef[i] = np.argmin(distances)
    
    return predRef


In [15]:
predRef = getPrediction(trainData, trainRef,testData)
print(f"acc : {accuracy(testRef,predRef):.2f}")


acc : 93.06


## Metody transformace příznaků
### Transformace na základě dekorelace (a následná redukce): PCA
- Z dat $X$ určíme kovariační matici $\Sigma_x$
- Vypočteme její vlastní vektory $e$ a sestavíme z nich matici $E$
- Pak transformovaná data:

$$X_{tr} = X (E_{0:f})^T$$
kde f je počet příznaků
- **Kovariační matice transformovaných dat bude diagonální**
- **Data v novém souřadném systému budou dekorelovaná**


### Transformace s ohledem na co největší diskriminativnost: LDA
Vypočteme matici vlastních vektorů $E$ z matice určené součinem $ \Sigma_{bc} \Sigma_{wc}^{-1} $

kde 
- kovarianční matice spočítaná ze středních hodnot tříd: $$\Sigma_{bc} = \frac{1}{N} \sum_{class=0}^{C} N_{class} (\mu_{class} - \mu)^T (\mu_{class} - \mu)$$ 
- průměrná kovarianční matice tříd: $$\Sigma_{wc} = \frac{1}{N} \sum_{class=0}^{C}{N_{class} \Sigma_{class}}$$ 


kde N je počet prvků a C je počet tříd

- Pak transformovaná data:

$$X_{tr} = X (E_{0:f})^T$$

In [ ]:

class transform:
    def __init__ (self, trainData, trainRef, testData, testRef):
        """
        """
        self.trainData = trainData
        self.testData = testData
        
        self.trainRef = trainRef
        self.testRef = testRef
        
        
    
    def pca(self, data, nFeautures=1):
        """
        transformuje data pca transformacni matici
        """
        cov = np.cov(data.T)
        eigen_values, eigen_vectors = np.linalg.eig(cov)
        sorted_idx = np.argsort(eigen_values)[::-1]
        new_data = data @ eigen_vectors[:, sorted_idx[:nFeautures]]
        
        return new_data
    
    
    def lda(self, data, nFeautures=1, train=True):    
        """
        transformuje data lda transformacni matici
        """
        
        refData = self.trainRef if train else self.testRef
        
        mean_data = np.mean(data, axis=0)
        classes = np.unique(refData)
        n_classes = classes.shape[0]
        cov_bc = np.zeros((data.shape[1], data.shape[1]))
        for c in classes:
            class_data = data[refData[:,0]==c]
            mean_class = np.mean(class_data, axis=0)
            n_class_samples = class_data.shape[0]
            cov_bc += n_class_samples * (mean_class - mean_data) @ (mean_class - mean_data).T
        cov_bc /= (n_classes - 1)
        # print("MC", mean_class.shape)
        # print("MD", mean_data.shape)
        # print("BC", cov_bc.shape)
        cov_wc = np.zeros((data.shape[1], data.shape[1]))
        for c in classes:
            class_data = data[refData[:,0]==c]
            mean_class =  np.mean(class_data, axis=0)
            mean_diff = class_data - mean_class
            cov_wc += mean_diff.T @ mean_diff
        cov_wc /= (data.shape[0] - n_classes)
        # print("WC", cov_wc.shape)
        cov = np.linalg.pinv(cov_wc) @ cov_bc
        eigen_values, eigen_vectors = np.linalg.eig(cov)
        sorted_idx = np.argsort(eigen_values)[::-1]
        new_data = data @ eigen_vectors[:, sorted_idx[:nFeautures]]
        
        return new_data


In [161]:
transformation = transform(trainData, trainRef, testData, testRef) 


In [152]:
#pca
for dim in range(1,np.size(trainData,1)+1):
    trainDataT = transformation.pca(trainData, dim)
    testDataT = transformation.pca(testData, dim)
    
    predRef = getPrediction(trainDataT,trainRef,testDataT)
    print(f"priznaky: {dim} -> acc : {accuracy(testRef,predRef):.2f} %")

priznaky: 1 -> acc : 66.94 %
priznaky: 2 -> acc : 88.35 %
priznaky: 3 -> acc : 93.06 %
priznaky: 4 -> acc : 93.29 %
priznaky: 5 -> acc : 93.53 %


In [162]:
#lda
for dim in range(1,np.size(trainData,1)+1):
    trainDataT = transformation.lda(trainData, dim)
    testDataT = transformation.lda(testData, dim, train=False)
    
    predRef = getPrediction(trainDataT,trainRef,testDataT)
    print(f"priznaky: {dim} -> acc : {accuracy(testRef,predRef):.2f} %")
  

priznaky: 1 -> acc : 33.41 %
priznaky: 2 -> acc : 29.65 %
priznaky: 3 -> acc : 41.41 %
priznaky: 4 -> acc : 24.24 %
priznaky: 5 -> acc : 26.00 %


In [ ]:
#Závěr:
#Kdy použít kterou metodu?
